# Solemne Optimización — Resolución con AMPL (amplpy)

Modelos AMPL resueltos desde Python con **amplpy**. Ejecuta todas las celdas en orden.
Al final se genera y descarga **`resultados_solemne.txt`** con toda la salida.

_Desarrollado con apoyo de inteligencia artificial (Claude, Anthropic)._


## 0. Instalación, configuración y buffer de salida

In [ ]:
%pip install -q amplpy
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(modules=["highs", "cbc"], license_uuid="default")

# Buffer: todo lo que pase por log() se imprime Y se guarda para el .txt final
REPORTE = []
def log(s=""):
    s = str(s)
    print(s)
    REPORTE.append(s)


## 1. Parte 1 — MILP determinista (ruteo)

In [ ]:
%%writefile p1_routing.mod
# =====================================================================
#  p1_routing.mod  ──  Parte 1: MILP determinista (ENTREGABLE AMPL)
#  Alineado 1:1 con p1_routing.py / datos.py.
#  Variables: r, z, w, u, x, q, arr, sal, FR, sh.
# =====================================================================

# ---------- Conjuntos ----------
set NODOS;                       # 0 = depósito, 1..4 = estaciones
set CLI within NODOS;            # estaciones (clientes)
set K;                           # camiones
set COMP;                        # compartimentos {0,1}
set PROD;                        # productos {R,D}
set ARCS := {i in NODOS, j in NODOS: i <> j};

# ---------- Parámetros ----------
param cd;                        # $/km
param cs;                        # $/L de faltante
param F {K};                     # costo fijo del camión
param Q {K,COMP};                # capacidad de compartimento [L]
param Dem {CLI,PROD};            # demanda [L]
param a {CLI};                   # apertura ventana [min]
param b {CLI};                   # cierre ventana   [min]
param tserv;                     # tiempo de servicio [min]
param v;                         # velocidad [km/min]
param dep {K};                   # salida del depósito [min]
param dist {ARCS};               # distancia [km]
param DELTA;                     # tolerancia de estabilidad
param Mfr;                       # Big-M fill ratio (≈2)
param Mt;                        # Big-M tiempo (≈2000)
param t {(i,j) in ARCS} := dist[i,j] / v;     # tiempo de viaje [min]
param n := card(CLI);

# ---------- Variables ----------
var r {ARCS,K} binary;                 # arco (i,j) usado por k
var z {K} binary;                      # camión usado
var w {K,COMP,PROD} binary;            # producto p en compartimento (k,c)
var u {CLI,K} integer >= 0;            # posición MTZ
var x {K,COMP,PROD} >= 0;              # carga total [L]
var q {CLI,K,COMP,PROD} >= 0;          # entregado en j desde (k,c) [L]
var arr {CLI,K} >= 0;                  # hora de llegada [min]
var sal {CLI,K} >= 0;                  # hora de salida  [min]
var FR {K,COMP,NODOS} >= 0, <= 1;      # fill ratio al salir del nodo
var sh {CLI,PROD} >= 0;                # faltante [L]

# ---------- Función objetivo ----------
minimize Costo:
    cd * sum {(i,j) in ARCS, k in K} dist[i,j] * r[i,j,k]
  + sum {k in K} F[k] * z[k]
  + cs * sum {j in CLI, p in PROD} sh[j,p];

# ---------- (R) Conservación de flujo ----------
s.t. R1_sale {k in K}:   sum {l in CLI} r[0,l,k] = z[k];
s.t. R1_vuelve {k in K}: sum {j in CLI} r[j,0,k] = z[k];
s.t. R2_balance {j in NODOS, k in K}:
     sum {i in NODOS: i<>j} r[i,j,k] = sum {l in NODOS: l<>j} r[j,l,k];
s.t. R3_unCamion {j in CLI}:
     sum {k in K, i in NODOS: i<>j} r[i,j,k] <= 1;

# ---------- (S) Subtours MTZ ----------
s.t. S1 {j in CLI, l in CLI, k in K: j<>l}:
     u[j,k] - u[l,k] + n * r[j,l,k] <= n - 1;

# ---------- (D) Demanda con shortage ----------
s.t. D1 {j in CLI, p in PROD}:
     sum {k in K, c in COMP} q[j,k,c,p] + sh[j,p] = Dem[j,p];

# ---------- (C) Compatibilidad compartimento-producto ----------
s.t. C1 {k in K, c in COMP}: sum {p in PROD} w[k,c,p] <= 1;
s.t. C2 {j in CLI, k in K, c in COMP, p in PROD}: q[j,k,c,p] <= Q[k,c] * w[k,c,p];
s.t. C3 {j in CLI, k in K, c in COMP, p in PROD}:
     q[j,k,c,p] <= Q[k,c] * sum {i in NODOS: i<>j} r[i,j,k];

# ---------- (K) Capacidad ----------
s.t. K1 {k in K, c in COMP, p in PROD}: x[k,c,p] = sum {j in CLI} q[j,k,c,p];
s.t. K2 {k in K, c in COMP}: sum {p in PROD} x[k,c,p] <= Q[k,c];

# ---------- (E) Estabilidad de carga (fill ratio) ----------
s.t. E1 {k in K, c in COMP}:
     FR[k,c,0] = ( sum {p in PROD} x[k,c,p] ) / Q[k,c];
s.t. E2 {(i,j) in ARCS, k in K, c in COMP: j in CLI}:
     FR[k,c,j] >= FR[k,c,i] - ( sum {p in PROD} q[j,k,c,p] )/Q[k,c] - Mfr*(1 - r[i,j,k]);
s.t. E3 {(i,j) in ARCS, k in K, c in COMP: j in CLI}:
     FR[k,c,j] <= FR[k,c,i] - ( sum {p in PROD} q[j,k,c,p] )/Q[k,c] + Mfr*(1 - r[i,j,k]);
s.t. E4a {j in NODOS, k in K}: FR[k,0,j] - FR[k,1,j] <= DELTA;
s.t. E4b {j in NODOS, k in K}: FR[k,1,j] - FR[k,0,j] <= DELTA;

# ---------- (T) Ventanas de tiempo (con espera permitida) ----------
s.t. T1 {l in CLI, k in K}:
     arr[l,k] >= dep[k] + t[0,l] - Mt*(1 - r[0,l,k]);
s.t. T2 {j in CLI, l in CLI, k in K: j<>l}:
     arr[l,k] >= sal[j,k] + t[j,l] - Mt*(1 - r[j,l,k]);
s.t. T3a {j in CLI, k in K}: sal[j,k] >= arr[j,k] + tserv;
s.t. T3b {j in CLI, k in K}:
     sal[j,k] >= a[j] + tserv - Mt*(1 - sum {i in NODOS: i<>j} r[i,j,k]);
s.t. T4 {j in CLI, k in K}:
     arr[j,k] <= b[j] + Mt*(1 - sum {i in NODOS: i<>j} r[i,j,k]);


In [ ]:
%%writefile p1_routing.dat
# =====================================================================
#  p1_routing.dat  ──  Datos Parte 1 (escenario s=1).  Mismo contenido
#  que datos.py.  Es "el .dat" del entregable AMPL.
# =====================================================================

set NODOS := 0 1 2 3 4;
set CLI   := 1 2 3 4;
set K     := T1 T2;
set COMP  := 0 1;
set PROD  := R D;

param cd    := 2;
param cs    := 10;
param tserv := 30;
param v     := 1;        # 60 km/h = 1 km/min
param DELTA := 0.30;
param Mfr   := 2;
param Mt    := 2000;

param F   := T1 500   T2 400;
param dep := T1 300   T2 330;     # 05:00 y 05:30 en minutos

param Q :   0      1 :=
   T1     8000   7000
   T2     6000   9000 ;

param Dem :   R      D :=
   1       3000   2000
   2       4000   1500
   3       2500   3000
   4       1000   2500 ;

param a := 1 360  2 420  3 480  4 360 ;
param b := 1 600  2 720  3 840  4 660 ;

param dist :=
 0 1 20  0 2 35  0 3 15  0 4 40
 1 0 20  2 0 35  3 0 15  4 0 40
 1 2 10  1 3 25  1 4 30
 2 1 10  3 1 25  4 1 30
 2 3 20  2 4 15  3 4 30
 3 2 20  4 2 15  4 3 30 ;


In [ ]:
ampl.reset()
ampl.read("p1_routing.mod")
ampl.read_data("p1_routing.dat")
ampl.option["solver"] = "highs"
log("#"*64); log("PARTE 1 - MILP determinista (AMPL/HiGHS)"); log("#"*64)
log(ampl.get_output("solve;"))
log(ampl.get_output(r'''
printf "Costo optimo: $%.0f\n", Costo;
printf "  distancia=$%.0f   fijo=$%.0f   shortage=%.0f L\n",
   cd * sum{(i,j) in ARCS,k in K} dist[i,j]*r[i,j,k], sum{k in K} F[k]*z[k],
   sum{j in CLI,p in PROD} sh[j,p];
printf "\nArcos usados:\n";
for {k in K, (i,j) in ARCS: r[i,j,k] > 0.5} printf "  %s: %d -> %d\n", k, i, j;
printf "\nCarga por compartimento:\n";
for {k in K, c in COMP, p in PROD: x[k,c,p] > 0.5}
   printf "  %s C%d: %.0f L de %s  (fill %.1f%%)\n", k, c, x[k,c,p], p, 100*x[k,c,p]/Q[k,c];
printf "\nEntregas q[j,k,c,p]:\n";
for {j in CLI, k in K, c in COMP, p in PROD: q[j,k,c,p] > 0.5}
   printf "  Est%d <- %s C%d: %.0f L de %s\n", j, k, c, q[j,k,c,p], p;
'''))


## 2. Parte 2 — Scheduling de carga (makespan)

In [ ]:
%%writefile p2_sched.mod
# =====================================================================
#  p2_sched.mod  ──  Parte 2: scheduling de carga (ENTREGABLE AMPL)
#  Minimizar el makespan con precedencia por camión y una bahía por producto.
#  Alineado 1:1 con p2_sched.py / datos_p2.py.
# =====================================================================

set OPS;                         # operaciones de carga: T1C0, T1C1, T2C0, T2C1
set PREC within {OPS, OPS};      # (i,j): i precede a j dentro del mismo camión (C0 antes de C1)
set CONF within {OPS, OPS};      # (i,j): comparten la misma bahía (no pueden solaparse)

param p {OPS};                   # tiempo de proceso (carga + limpieza) [min]
param M;                         # Big-M

var s {OPS} >= 0;                # tiempo de inicio de carga [min]
var Cmax >= 0;                   # makespan [min]
var y {CONF} binary;             # 1 si i precede a j en la bahía compartida

minimize Makespan: Cmax;

# Precedencia dentro del camión: C1 empieza cuando C0 termina
s.t. Precedencia {(i,j) in PREC}: s[j] >= s[i] + p[i];

# No-solapamiento en la misma bahía (disyuntiva big-M)
s.t. Bahia_a {(i,j) in CONF}: s[j] >= s[i] + p[i] - M*(1 - y[i,j]);
s.t. Bahia_b {(i,j) in CONF}: s[i] >= s[j] + p[j] - M*y[i,j];

# Makespan
s.t. Cota {o in OPS}: Cmax >= s[o] + p[o];


In [ ]:
%%writefile p2_sched.dat
# =====================================================================
#  p2_sched.dat  ──  Datos Parte 2 (volúmenes fijos s=1)
#  p[k,c] = VL/tasa + limpieza:
#    T1C0=5000/500+5=15   T1C1=5000/400+5=17.5
#    T2C0=4000/400+5=15   T2C1=3500/500+5=12
# =====================================================================

set OPS := T1C0 T1C1 T2C0 T2C1;

param p :=
   T1C0  15
   T1C1  17.5
   T2C0  15
   T2C1  12 ;

param M := 100;

# Precedencia dentro del camión (C0 antes de C1)
set PREC := (T1C0,T1C1) (T2C0,T2C1) ;

# Operaciones que comparten bahía:
#   bahía Regular: T1C0 y T2C1   |   bahía Diésel: T1C1 y T2C0
set CONF := (T1C0,T2C1) (T1C1,T2C0) ;


In [ ]:
ampl.reset()
ampl.read("p2_sched.mod")
ampl.read_data("p2_sched.dat")
ampl.option["solver"] = "highs"
log("\n"+"#"*64); log("PARTE 2 - Scheduling de carga"); log("#"*64)
log(ampl.get_output("solve;"))
log(ampl.get_output(r'''
printf "Makespan = %.1f min\n", Cmax;
printf "Programa de carga (inicio -> fin):\n";
for {o in OPS} printf "  %s: %.1f -> %.1f\n", o, s[o], s[o] + p[o];
'''))


## 3. Parte 3 — Verificación de la solución del operador

In [ ]:
log("\n"+"#"*64); log("PARTE 3 - Verificacion + mejora"); log("#"*64)
D = {(1,"R"):3000,(1,"D"):2000,(2,"R"):4000,(2,"D"):1500,
     (3,"R"):2500,(3,"D"):3000,(4,"R"):1000,(4,"D"):2500}
Q = {("T1",0):8000,("T1",1):7000,("T2",0):6000,("T2",1):9000}
dist = {(0,1):20,(0,2):35,(0,3):15,(0,4):40,(1,2):10,(1,3):25,(1,4):30,(2,3):20,(2,4):15,(3,4):30}
dist.update({(b,a):v for (a,b),v in list(dist.items())})
DELTA = 0.30
def verificar(nombre, sol):
    log("="*56); log(nombre); log("="*56)
    fact = True
    for k, info in sol.items():
        est = [j for j in info["route"] if j != 0]
        rem = {c: sum(D[(j, info["comp"][c])] for j in est) for c in (0,1)}
        for c in (0,1):
            if rem[c] > Q[(k,c)]: fact=False; log(f"  {k} C{c} EXCEDE capacidad")
        log(f" {k}: ruta {info['route']}")
        for idx, nodo in enumerate(info["route"][:-1]):
            if idx > 0:
                for c in (0,1): rem[c] -= D[(nodo, info["comp"][c])]
            fr = {c: rem[c]/Q[(k,c)] for c in (0,1)}; dif = abs(fr[0]-fr[1])
            etq = "salida" if idx==0 else f"tras Est{nodo}"; ok = dif <= DELTA + 1e-9; fact &= ok
            log(f"    {etq:<12} C0={fr[0]*100:5.1f}%  C1={fr[1]*100:5.1f}%  |dif|={dif:.3f}  {'OK' if ok else 'VIOLA'}")
    km = sum(dist[(r[i],r[i+1])] for info in sol.values() for r in [info["route"]] for i in range(len(r)-1))
    costo = 2*km + sum(500 if k=="T1" else 400 for k in sol)
    log(f" Distancia={km} km -> costo real = ${costo}  >>> " + ("FACTIBLE" if fact else "INFACTIBLE")); log("")
operador = {"T1":{"route":[0,1,3,0],"comp":{0:"R",1:"D"}}, "T2":{"route":[0,2,4,0],"comp":{0:"D",1:"R"}}}
mejora   = {"T1":{"route":[0,1,2,4,0],"comp":{0:"R",1:"D"}}, "T2":{"route":[0,3,0],"comp":{0:"D",1:"R"}}}
verificar("OPERADOR (declara $260, shortage 0)", operador)
verificar("MEJORA (optimo P1)", mejora)


## 4. Parte 4 — Estocástico de dos etapas

In [ ]:
%%writefile p4_stoch.mod
# =====================================================================
#  p4_stoch.mod  ──  Parte 4: estocástico de DOS ETAPAS (ENTREGABLE AMPL)
#  Forma extensiva (todos los escenarios juntos).
#  1ª etapa: r, z, w, u, x, arr, sal   (no dependen del escenario)
#  2ª etapa: q[s], sh[s], FR[s]        (recurso, una copia por escenario)
# =====================================================================

# ---------- Conjuntos ----------
set NODOS;  set CLI within NODOS;  set K;  set COMP;  set PROD;
set S;                                     # escenarios
set ARCS := {i in NODOS, j in NODOS: i <> j};

# ---------- Parámetros ----------
param cd;  param cs;  param tserv;  param v;  param DELTA;  param Mfr;  param Mt;
param F {K};  param Q {K,COMP};  param dep {K};
param a {CLI};  param b {CLI};   param dist {ARCS};
param prob {S};                            # probabilidad del escenario
param Dem {S,CLI,PROD};                    # demanda por escenario
param t {(i,j) in ARCS} := dist[i,j]/v;
param n := card(CLI);

# ---------- Variables de 1ª etapa ----------
var r {ARCS,K} binary;
var z {K} binary;
var w {K,COMP,PROD} binary;
var u {CLI,K} integer >= 0;
var x {K,COMP,PROD} >= 0;                   # CARGA (se decide antes de la demanda)
var arr {CLI,K} >= 0;
var sal {CLI,K} >= 0;

# ---------- Variables de 2ª etapa (recurso, por escenario) ----------
var q  {S,CLI,K,COMP,PROD} >= 0;           # entregas por escenario
var sh {S,CLI,PROD} >= 0;                   # faltante por escenario
var FR {S,K,COMP,NODOS} >= 0, <= 1;

# ---------- Objetivo: 1ª etapa + faltante ESPERADO ----------
minimize CostoEsperado:
    cd * sum {(i,j) in ARCS, k in K} dist[i,j]*r[i,j,k]
  + sum {k in K} F[k]*z[k]
  + sum {s in S} prob[s] * cs * sum {j in CLI, p in PROD} sh[s,j,p];

# ===================== 1ª ETAPA =====================
s.t. R1_sale {k in K}:   sum {l in CLI} r[0,l,k] = z[k];
s.t. R1_vuelve {k in K}: sum {j in CLI} r[j,0,k] = z[k];
s.t. R2 {j in NODOS, k in K}:
     sum {i in NODOS: i<>j} r[i,j,k] = sum {l in NODOS: l<>j} r[j,l,k];
s.t. R3 {j in CLI}: sum {k in K, i in NODOS: i<>j} r[i,j,k] <= 1;
s.t. S1 {j in CLI, l in CLI, k in K: j<>l}: u[j,k]-u[l,k]+n*r[j,l,k] <= n-1;

s.t. C1 {k in K, c in COMP}: sum {p in PROD} w[k,c,p] <= 1;
s.t. Cap {k in K, c in COMP}: sum {p in PROD} x[k,c,p] <= Q[k,c];
s.t. CargaProd {k in K, c in COMP, p in PROD}: x[k,c,p] <= Q[k,c]*w[k,c,p];

s.t. T1 {l in CLI, k in K}: arr[l,k] >= dep[k] + t[0,l] - Mt*(1 - r[0,l,k]);
s.t. T2 {j in CLI, l in CLI, k in K: j<>l}: arr[l,k] >= sal[j,k] + t[j,l] - Mt*(1 - r[j,l,k]);
s.t. T3a {j in CLI, k in K}: sal[j,k] >= arr[j,k] + tserv;
s.t. T3b {j in CLI, k in K}: sal[j,k] >= a[j] + tserv - Mt*(1 - sum {i in NODOS: i<>j} r[i,j,k]);
s.t. T4 {j in CLI, k in K}: arr[j,k] <= b[j] + Mt*(1 - sum {i in NODOS: i<>j} r[i,j,k]);

# ===================== 2ª ETAPA (por escenario) =====================
s.t. Demanda {s in S, j in CLI, p in PROD}:
     sum {k in K, c in COMP} q[s,j,k,c,p] + sh[s,j,p] = Dem[s,j,p];
s.t. NoMasQueCargado {s in S, k in K, c in COMP}:
     sum {j in CLI, p in PROD} q[s,j,k,c,p] <= sum {p in PROD} x[k,c,p];
s.t. SoloProd {s in S, j in CLI, k in K, c in COMP, p in PROD}: q[s,j,k,c,p] <= Q[k,c]*w[k,c,p];
s.t. SoloVisita {s in S, j in CLI, k in K, c in COMP, p in PROD}:
     q[s,j,k,c,p] <= Q[k,c] * sum {i in NODOS: i<>j} r[i,j,k];

s.t. E1 {s in S, k in K, c in COMP}: FR[s,k,c,0] = ( sum {p in PROD} x[k,c,p] )/Q[k,c];
s.t. E2 {s in S, (i,j) in ARCS, k in K, c in COMP: j in CLI}:
     FR[s,k,c,j] >= FR[s,k,c,i] - ( sum {p in PROD} q[s,j,k,c,p] )/Q[k,c] - Mfr*(1 - r[i,j,k]);
s.t. E3 {s in S, (i,j) in ARCS, k in K, c in COMP: j in CLI}:
     FR[s,k,c,j] <= FR[s,k,c,i] - ( sum {p in PROD} q[s,j,k,c,p] )/Q[k,c] + Mfr*(1 - r[i,j,k]);
s.t. E4a {s in S, j in NODOS, k in K}: FR[s,k,0,j] - FR[s,k,1,j] <= DELTA;
s.t. E4b {s in S, j in NODOS, k in K}: FR[s,k,1,j] - FR[s,k,0,j] <= DELTA;


In [ ]:
%%writefile p4_stoch.dat
# =====================================================================
#  p4_stoch.dat  ──  Datos Parte 4 (3 escenarios)
#  Supuestos: Est.4 fija en todo s; Est.3 Diésel = 3000 en todo s.
# =====================================================================

set NODOS := 0 1 2 3 4;
set CLI   := 1 2 3 4;
set K     := T1 T2;
set COMP  := 0 1;
set PROD  := R D;
set S     := 1 2 3;

param cd    := 2;
param cs    := 10;
param tserv := 30;
param v     := 1;
param DELTA := 0.30;
param Mfr   := 2;
param Mt    := 2000;

param F   := T1 500   T2 400;
param dep := T1 300   T2 330;
param prob := 1 0.5   2 0.3   3 0.2;

param Q :   0      1 :=
   T1     8000   7000
   T2     6000   9000 ;

param a := 1 360  2 420  3 480  4 360 ;
param b := 1 600  2 720  3 840  4 660 ;

# Demanda por escenario  Dem[s,j,p]
param Dem :=
 [1,*,*]:   R     D :=
        1  3000  2000
        2  4000  1500
        3  2500  3000
        4  1000  2500
 [2,*,*]:   R     D :=
        1  4500  2800
        2  5500  2000
        3  3500  3000
        4  1000  2500
 [3,*,*]:   R     D :=
        1  2000  1200
        2  3000  1000
        3  1800  3000
        4  1000  2500 ;

param dist :=
 0 1 20  0 2 35  0 3 15  0 4 40
 1 0 20  2 0 35  3 0 15  4 0 40
 1 2 10  1 3 25  1 4 30
 2 1 10  3 1 25  4 1 30
 2 3 20  2 4 15  3 4 30
 3 2 20  4 2 15  4 3 30 ;


In [ ]:
ampl.reset()
ampl.read("p4_stoch.mod")
ampl.read_data("p4_stoch.dat")
ampl.option["solver"] = "highs"
log("\n"+"#"*64); log("PARTE 4 - Estocastico de dos etapas"); log("#"*64)
log(ampl.get_output("solve;"))
log(ampl.get_output(r'''
printf "RP (costo esperado) = $%.1f\n", CostoEsperado;
printf "  1a etapa (dist+fijo)=$%.0f   faltante esperado=$%.1f\n",
   cd * sum{(i,j) in ARCS,k in K} dist[i,j]*r[i,j,k] + sum{k in K} F[k]*z[k],
   sum{s in S} prob[s]*cs*sum{j in CLI,p in PROD} sh[s,j,p];
printf "\nRutas (1a etapa):\n";
for {k in K, (i,j) in ARCS: r[i,j,k] > 0.5} printf "  %s: %d -> %d\n", k, i, j;
printf "\nFaltante por escenario:\n";
for {s in S} printf "  s=%d: %.0f L\n", s, sum{j in CLI,p in PROD} sh[s,j,p];
'''))


## 5. Generar y descargar `resultados_solemne.txt`

In [ ]:
with open("resultados_solemne.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(REPORTE))
print(">>> Guardado: resultados_solemne.txt")
try:
    from google.colab import files
    files.download("resultados_solemne.txt")   # descarga automatica en Colab
except Exception:
    print("(Si no estas en Colab, descargalo desde el panel de Archivos.)")
